# Spike — validação do NotificadorEmail

Testa em 2 etapas: (1) leitura segura da senha via dbutils.secrets.get(), confirmando redação automática; (2) envio real de 1 email de teste via Gmail SMTP.

Referências: ADR-007 (desenho original de alertas, escopo revisado).

In [0]:
senha_teste = dbutils.secrets.get(scope="pulse-secrets", key="gmail-app-password")

print("Tipo do valor:", type(senha_teste))
print("Tamanho:", len(senha_teste))
print("Tentando print direto:", senha_teste)

In [0]:
import smtplib
from email.mime.text import MIMEText

senha = dbutils.secrets.get(scope="pulse-secrets", key="gmail-app-password")
remetente = "bruno.queles.dataeng@gmail.com"

corpo = MIMEText("Teste de envio real via NotificadorEmail — spike de validação, Pulse Health Platform.")
corpo["Subject"] = "[Pulse Health Platform] Teste de envio real"
corpo["From"] = remetente
corpo["To"] = remetente

try:
    with smtplib.SMTP("smtp.gmail.com", 587) as servidor:
        servidor.starttls()
        servidor.login(remetente, senha)
        servidor.sendmail(remetente, [remetente], corpo.as_string())
    print("Email enviado com sucesso!")
except Exception as e:
    print(f"Falha ao enviar: {type(e).__name__} - {e}")

In [0]:
from src.observabilidade.notificadores import NotificadorEmail

notificador_email = NotificadorEmail(dbutils=dbutils)

alerta_teste = {
    "tipo_evento": "teste_classe",
    "origem": "spike_notificador_email",
    "severidade": "baixa",
    "mensagem": "Teste da classe NotificadorEmail (não mais código solto no spike)",
    "detalhes": {"validacao": "camada 3 pendente ainda, isso e so a classe isolada"},
}

resultado = notificador_email.notificar(alerta_teste)
print("Sucesso:", resultado)